In [5]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# 1. Automatic dataset download (UCI Heart Disease)
url = "https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv"
df = pd.read_csv(url)

print(f"Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")

# Ensure target column is named 'target'
if 'target' not in df.columns:
    df.rename(columns={df.columns[-1]: 'target'}, inplace=True)

# 2. Train-test split (80% train, 20% test)
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Create folders and export test_data.csv
os.makedirs("model", exist_ok=True)
test_df = X_test.copy()
test_df['target'] = y_test.values
test_df.to_csv("test_data.csv", index=False)
print("-> test_data.csv saved successfully!")

# 4. Define the 5 Classifiers using Pipelines (includes automatic scaling)
models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    "KNN": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', KNeighborsClassifier(n_neighbors=7))
    ]),
    "Naive Bayes": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', GaussianNB())
    ]),
    "Support Vector Machine": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', CalibratedClassifierCV(
            estimator=SVC(kernel='rbf', random_state=42),
            ensemble=False
        ))
    ])
}

# 5. Train, evaluate on test set, and save models
metrics_summary = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    
    # Save model artifact into model/
    filename = os.path.join("model", f"{name.lower().replace(' ', '_')}.joblib")
    joblib.dump(pipe, filename)
    
    # Evaluate
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    metrics_summary.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

print("\n--- MODEL METRICS TABLE ---")
print(pd.DataFrame(metrics_summary).to_string(index=False))

Dataset loaded successfully: 303 rows, 14 columns
-> test_data.csv saved successfully!

--- MODEL METRICS TABLE ---
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.8033 0.8690     0.7692  0.9091 0.8333 0.6098
         Decision Tree    0.7869 0.8176     0.7500  0.9091 0.8219 0.5789
                   KNN    0.8197 0.8837     0.7895  0.9091 0.8451 0.6410
           Naive Bayes    0.8197 0.8755     0.7895  0.9091 0.8451 0.6410
Support Vector Machine    0.8361 0.8831     0.7949  0.9394 0.8611 0.6784
